# Preprocessing
This notebook will be used for preprocessing steps.

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [5]:
# ── Data path ────────────────────────────────────────────────────
# CSVs are stored locally and gitignored. See data-loading-guide.md.
DATA_DIR = Path.home()  # adjust if moved

FILE_MONTHS = [
    'CRMLSSold202511.csv',   # November 2025
    'CRMLSSold202512.csv',   # December 2025
    'CRMLSSold202601.csv',   # January 2026
    'CRMLSSold202602.csv',   # February 2026
    'CRMLSSold202603.csv',   # March 2026
    'CRMLSSold202604.csv',   # April 2026
    'CRMLSSold202605.csv',   # May 2026  <- held-out test month
]

frames = []
for fname in FILE_MONTHS:
    df_month = pd.read_csv(DATA_DIR / fname, low_memory=False)
    df_month['_source_file'] = fname
    frames.append(df_month)

raw = pd.concat(frames, ignore_index=True)
print(f'Total rows loaded (all PropertyTypes): {len(raw):,}')

Total rows loaded (all PropertyTypes): 143,492


In [ ]:
# ── Apply project filter ─────────────────────────────────────────
df = raw[
    (raw['PropertyType'] == 'Residential') &
    (raw['PropertySubType'] == 'SingleFamilyResidence')
].copy()

print(f'Rows after SFR filter: {len(df):,}')

# Parse dates
for col in ['CloseDate', 'ListingContractDate', 'PurchaseContractDate']:
    sfr[col] = pd.to_datetime(df[col], errors='coerce')

df['CloseYearMonth'] = df['CloseDate'].dt.to_period('M')
df['CloseMonth']     = df['CloseDate'].dt.month

---
## 1. Drop Fully-Null & Unusable Fields

Per the Week 2 field audit, these Trestle fields are 100% null in this CRMLS extract, or are
otherwise unusable (freeform text, redundant, or too sparse to impute reliably). They are
dropped before any further processing.

In [ ]:
# Fields confirmed 100% null in this extract (Week 2 audit)
FULLY_NULL_FIELDS = [
    'AboveGradeFinishedArea',
    'BelowGradeFinishedArea',
    'BuildingAreaTotal',
    'TaxAnnualAmount',
    'CoveredSpaces',
    'FireplacesTotal',          # use FireplaceYN instead
    'BusinessType',             # not applicable to SFR
    'ElementarySchoolDistrict',
    'MiddleOrJuniorSchoolDistrict',
]

# Additional fields dropped for sparsity / redundancy (not 100% null, but unusable)
SPARSE_OR_REDUNDANT_FIELDS = [
    'WaterfrontYN',             # 99.9% null
    'BasementYN',               # 97.6% null
    'ElementarySchool',         # 87.3% null
    'MiddleOrJuniorSchool',     # 87.2% null
    'HighSchool',               # 83.2% null -- HighSchoolDistrict preferred
    'LotSizeAcres',             # redundant with LotSizeSquareFeet
    'LotSizeArea',              # redundant with LotSizeSquareFeet
    'LotSizeDimensions',        # freeform String 150 -- not modelable
    'MainLevelBedrooms',        # 39.0% null; overlaps BedroomsTotal
]

DROP_FIELDS = FULLY_NULL_FIELDS + SPARSE_OR_REDUNDANT_FIELDS
DROP_FIELDS = [c for c in DROP_FIELDS if c in df.columns]

df.drop(columns=DROP_FIELDS, inplace=True)
print(f'Dropped {len(DROP_FIELDS)} fields.')
print(f'Remaining columns: {df.shape[1]}')

---
## 2. Drop Structural Outliers

Per the Week 2 outlier deep dive, a small number of records have data quality issues that
make them unsuitable for training (not legitimate market signal). These rows are dropped
outright, separately from the imputation strategy in the next section.

In [ ]:
before = len(df)

# ClosePrice < $10k -- confirmed likely data entry errors
df = df[df['ClosePrice'] >= 10_000]

# LivingArea == 0 -- structural data error, cannot compute PricePerSqFt or size features
df = df[df['LivingArea'] > 0]

after = len(df)
print(f'Rows before outlier drop: {before:,}')
print(f'Rows after outlier drop:  {after:,}')
print(f'Rows dropped:             {before - after}')

# NOTE: ClosePrice > $10M (288 records) and BedroomsTotal == 0 (31 records) are
# NOT dropped here -- they are legitimate (if rare) transactions. They will be
# addressed via log-transformation (price) and left as-is (bedrooms) rather than removal.

---
## 3. Handle Missing Values

Each remaining field with nulls is handled per the Week 2 field audit decision table.
Strategy is one of: **impute** (fill with a sensible default), **flag** (create a
missing-indicator column, useful when missingness itself may be informative), or
**leave for Week 6** (school district, to be filled via spatial join).

In [ ]:
# ── Boolean amenity fields -- impute False ───────────────────────
# NaN in these Trestle Boolean fields most plausibly means the feature is absent
# (agents tend to actively flag amenities that ARE present, not the reverse)
BOOLEAN_IMPUTE_FALSE = [
    'PoolPrivateYN', 'ViewYN', 'FireplaceYN',
    'AttachedGarageYN', 'NewConstructionYN',
]

for col in BOOLEAN_IMPUTE_FALSE:
    if col in df.columns:
        n_null = df[col].isna().sum()
        df[col] = df[col].fillna(False)
        print(f'{col}: imputed {n_null:,} nulls -> False')

In [ ]:
# ── GarageSpaces -- impute 0 (no garage) ─────────────────────────
n_null = df['GarageSpaces'].isna().sum()
df['GarageSpaces'] = df['GarageSpaces'].fillna(0)
print(f'GarageSpaces: imputed {n_null:,} nulls -> 0')

# ── ParkingTotal -- impute 0 ──────────────────────────────────────
if 'ParkingTotal' in df.columns:
    n_null = df['ParkingTotal'].isna().sum()
    df['ParkingTotal'] = df['ParkingTotal'].fillna(0)
    print(f'ParkingTotal: imputed {n_null:,} nulls -> 0')

In [ ]:
# ── AssociationFee / AssociationFeeFrequency -- impute no-HOA ────
# NaN AssociationFee (28.6% null) most plausibly means no HOA exists
n_null = df['AssociationFee'].isna().sum()
df['AssociationFee'] = df['AssociationFee'].fillna(0)
df['AssociationFeeFrequency'] = df['AssociationFeeFrequency'].fillna('None')
print(f'AssociationFee: imputed {n_null:,} nulls -> 0')
print(f'AssociationFeeFrequency: nulls -> \'None\'')

In [ ]:
# ── LotSizeSquareFeet -- impute with ZIP-level median ────────────
# Falls back to county median, then global median, for ZIPs with no other data
n_null = df['LotSizeSquareFeet'].isna().sum()

zip_median = df.groupby('PostalCode')['LotSizeSquareFeet'].transform('median')
county_median = df.groupby('CountyOrParish')['LotSizeSquareFeet'].transform('median')
global_median = df['LotSizeSquareFeet'].median()

df['LotSizeSquareFeet'] = (
    df['LotSizeSquareFeet']
    .fillna(zip_median)
    .fillna(county_median)
    .fillna(global_median)
)
print(f'LotSizeSquareFeet: imputed {n_null:,} nulls (ZIP median -> county median -> global median)')

In [ ]:
# ── Stories -- impute with mode ──────────────────────────────────
n_null = df['Stories'].isna().sum()
stories_mode = df['Stories'].mode()[0]
df['Stories'] = df['Stories'].fillna(stories_mode)
print(f'Stories: imputed {n_null:,} nulls -> mode ({stories_mode})')

In [ ]:
# ── YearBuilt -- impute with median (0.1% null, negligible impact) ──
n_null = df['YearBuilt'].isna().sum()
year_median = df['YearBuilt'].median()
df['YearBuilt'] = df['YearBuilt'].fillna(year_median)
print(f'YearBuilt: imputed {n_null:,} nulls -> median ({year_median:.0f})')

In [ ]:
# ── Flooring -- impute 'Unknown' (36.3% null) ────────────────────
n_null = df['Flooring'].isna().sum()
df['Flooring'] = df['Flooring'].fillna('Unknown')
print(f'Flooring: imputed {n_null:,} nulls -> \'Unknown\'')

In [ ]:
# ── Levels -- impute with mode ───────────────────────────────────
n_null = df['Levels'].isna().sum()
levels_mode = df['Levels'].mode()[0]
df['Levels'] = df['Levels'].fillna(levels_mode)
print(f'Levels: imputed {n_null:,} nulls -> mode (\'{levels_mode}\')')

In [ ]:
# ── HighSchoolDistrict -- FLAG rather than impute (26.9% null) ───
# Do not guess a district. Create a missing-indicator column, and leave the
# null values in place -- Week 6's spatial join against CA School District
# Areas 2024-25 boundaries (via Latitude/Longitude) will fill these in properly.
df['HighSchoolDistrict_Missing'] = df['HighSchoolDistrict'].isna().astype(int)
print(f'HighSchoolDistrict: {df["HighSchoolDistrict_Missing"].sum():,} nulls flagged '
      f'(to be filled via Week 6 spatial join, not imputed here)')

In [ ]:
# ── MLSAreaMajor / City -- impute 'Unknown' for categorical fallback ──
for col in ['MLSAreaMajor', 'City']:
    if col in df.columns:
        n_null = df[col].isna().sum()
        if n_null > 0:
            df[col] = df[col].fillna('Unknown')
            print(f'{col}: imputed {n_null:,} nulls -> \'Unknown\'')

In [ ]:
# ── Final null check on remaining fields ─────────────────────────
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)

print('Fields still containing nulls after imputation:')
if len(remaining_nulls) == 0:
    print('  None -- all handled.')
else:
    display(remaining_nulls.to_frame('null_count'))

# NOTE: HighSchoolDistrict nulls remain intentionally (flagged, not imputed)

---
## 4. Feature Engineering

Derived features identified during Week 2 EDA. Each is engineered here from the now-clean
base fields, ready for encoding and modelling.

In [ ]:
# PropertyAge -- years since construction
df['PropertyAge'] = 2026 - df['YearBuilt']

# BedBathRatio -- layout shape signal
df['BedBathRatio'] = df['BedroomsTotal'] / df['BathroomsTotalInteger'].replace(0, np.nan)
df['BedBathRatio'] = df['BedBathRatio'].fillna(df['BedBathRatio'].median())

# SaleToListRatio -- exploratory / diagnostic, NOT a model input (uses ClosePrice directly = leakage)
df['SaleToListRatio'] = df['ClosePrice'] / df['ListPrice']

# PriceReductionYN -- signals seller motivation, uses only pre-sale info (no leakage)
df['PriceReductionYN'] = (df['ListPrice'] < df['OriginalListPrice']).astype(int)

# MonthlyHOA -- normalize AssociationFee to a monthly equivalent
FREQ_TO_MONTHLY = {
    'Monthly': 1, 'Quarterly': 1/3, 'Semi-Annually': 1/6, 'Annually': 1/12, 'None': 0,
}
df['MonthlyHOA'] = (
    df['AssociationFee'] *
    df['AssociationFeeFrequency'].map(FREQ_TO_MONTHLY).fillna(0)
)

# PricePerSqFt -- exploratory only; excluded from model inputs (see Section 5 note)
df['PricePerSqFt'] = df['ClosePrice'] / df['LivingArea'].replace(0, np.nan)

# Levels_primary -- first value of a multi-value Levels enum
df['Levels_primary'] = df['Levels'].str.split(',').str[0].str.strip()

print('Engineered features created:')
print('  PropertyAge, BedBathRatio, SaleToListRatio, PriceReductionYN,')
print('  MonthlyHOA, PricePerSqFt, Levels_primary')

In [ ]:
# Flooring binary flags -- top individual types with >= 200 listings (from Week 2 EDA)
flooring_exploded = df['Flooring'].str.split(',').explode().str.strip()
top_flooring_types = flooring_exploded.value_counts()
top_flooring_types = top_flooring_types[
    (top_flooring_types.index != 'Unknown') & (top_flooring_types >= 200)
].index.tolist()

for ft in top_flooring_types:
    col_name = f'Flooring_{ft.replace(" ", "_").replace("/", "_")}'
    df[col_name] = df['Flooring'].str.contains(ft, case=False, na=False).astype(int)

print(f'Created {len(top_flooring_types)} binary Flooring_* flags: {top_flooring_types}')

In [ ]:
# CloseMonth as a cyclical feature (captures seasonality without a false ordinal jump
# from December=12 to January=1)
df['CloseMonth_sin'] = np.sin(2 * np.pi * df['CloseMonth'] / 12)
df['CloseMonth_cos'] = np.cos(2 * np.pi * df['CloseMonth'] / 12)

print('CloseMonth_sin / CloseMonth_cos created for cyclical month encoding.')

---
## 5. Encode Categorical Fields to Numeric

Different encoding strategies are used depending on cardinality and structure:

| Field | Cardinality | Strategy |
|---|---|---|
| `Levels_primary` | 4 categories | Ordinal encoding (natural order: One < Two < ThreeOrMore) |
| `NewConstructionYN` etc. | Boolean | Already 0/1 after imputation |
| `CountyOrParish` | 60 | One-hot encoding (or target encoding for tree models) |
| `PostalCode` | 1,675 | Target (mean) encoding -- one-hot would explode dimensionality |
| `City` | ~928 | Target (mean) encoding -- same reasoning as ZIP |
| `HighSchoolDistrict` | Many, 26.9% null | Target encoding; nulls encoded separately (filled Week 6) |

Target encoding is computed **only on the training set** and mapped onto the test set,
to avoid leaking test-set price information into the encoding.

In [ ]:
# ── Ordinal encode Levels_primary ─────────────────────────────────
LEVELS_ORDER = {'One': 1, 'Two': 2, 'ThreeOrMore': 3, 'MultiSplit': 2}  # MultiSplit ~ 2 floors typical
df['Levels_encoded'] = df['Levels_primary'].map(LEVELS_ORDER).fillna(1).astype(int)

print('Levels_primary encoded ordinally:')
print(df[['Levels_primary', 'Levels_encoded']].drop_duplicates().sort_values('Levels_encoded'))

In [ ]:
# ── One-hot encode CountyOrParish (60 categories -- manageable) ──
county_dummies = pd.get_dummies(df['CountyOrParish'], prefix='County', drop_first=True)
print(f'CountyOrParish one-hot encoded into {county_dummies.shape[1]} columns.')
print(f'(Encoding will be concatenated to the final feature matrix in Section 8.)')

In [ ]:
# ── Target encoding for high-cardinality fields ───────────────────
# IMPORTANT: fit encoders on TRAIN only, apply to test, to prevent leakage.
# This function is defined here and invoked AFTER the train/test split (Section 7),
# so the split must happen before target encoding is finalized.

def fit_target_encoding(train_df, col, target='ClosePrice', min_samples=10, smoothing=10):
    '''
    Smoothed mean target encoding: blends each category's mean with the global mean,
    weighted by how many samples that category has. Prevents overfitting to rare
    categories (e.g. ZIP codes with only 1-2 sales).
    '''
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
    return smoothed, global_mean

def apply_target_encoding(df, col, encoding_map, global_mean):
    return df[col].map(encoding_map).fillna(global_mean)

print('Target encoding function defined -- applied to PostalCode, City, HighSchoolDistrict')
print('after the train/test split (Section 7), to avoid leakage.')

---
## 6. Normalize Numerical Features

Tree-based models (Random Forest, Gradient Boosting) are scale-invariant and do not require
normalization. However, the baseline **Linear Regression** model (Week 4) does benefit from
scaled features, particularly when regularization (Ridge/Lasso) is introduced later.

`StandardScaler` is fit on the **training set only** and applied to both train and test,
consistent with the target-encoding leakage precaution above. This is done after the
train/test split in Section 7 to keep the fit strictly within the training window.

In [ ]:
# Numerical features that will be scaled for linear models
# (tree-based models in Week 5/7 will use the unscaled versions)
NUMERIC_FEATURES_TO_SCALE = [
    'LivingArea', 'LotSizeSquareFeet', 'BedroomsTotal', 'BathroomsTotalInteger',
    'Stories', 'PropertyAge', 'GarageSpaces', 'ParkingTotal', 'DaysOnMarket',
    'ListPrice', 'BedBathRatio', 'MonthlyHOA',
]

print('Numerical features flagged for scaling (Linear Regression only):')
for f in NUMERIC_FEATURES_TO_SCALE:
    print(f'  {f}')

print()
print('Distributions before scaling:')
display(df[NUMERIC_FEATURES_TO_SCALE].describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
# Visual check: are any of these heavily skewed and in need of log-transformation
# BEFORE scaling (scaling alone does not fix skew)?
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
skew_check_cols = ['LivingArea', 'LotSizeSquareFeet', 'ListPrice',
                    'DaysOnMarket', 'MonthlyHOA', 'ClosePrice']

for ax, col in zip(axes.flat, skew_check_cols):
    data = df[col].clip(upper=df[col].quantile(0.99))
    data.plot(kind='hist', bins=50, ax=ax, color='steelblue', edgecolor='white')
    skewness = df[col].skew()
    ax.set_title(f'{col}\n(skew={skewness:.2f})', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('Fields with skew > 1.0 are candidates for log1p transformation before scaling.')

In [ ]:
# Apply log1p to heavily right-skewed features (skew > 1.0)
# ClosePrice (target) is log-transformed separately and the model predicts log-price
LOG_TRANSFORM_CANDIDATES = ['LivingArea', 'LotSizeSquareFeet', 'ListPrice', 'MonthlyHOA']

for col in LOG_TRANSFORM_CANDIDATES:
    skewness = df[col].skew()
    if skewness > 1.0:
        df[f'{col}_log'] = np.log1p(df[col])
        print(f'{col}: skew={skewness:.2f} -> log1p transform applied ({col}_log)')
    else:
        print(f'{col}: skew={skewness:.2f} -> no transform needed')

# Target variable log-transform (per Week 2 finding: ClosePrice is heavily right-skewed)
df['ClosePrice_log'] = np.log1p(df['ClosePrice'])
print(f'\nClosePrice: skew={df["ClosePrice"].skew():.2f} -> log1p transform applied (target for linear models)')

---
## 7. Train / Test Split

Per the task specification: **the most recent month is the test set**, and **X preceding
months form the training set**, where X is a tunable window length rather than fixed at 6.

This section builds a reusable function so that X can be experimented with in Week 4+
without re-running the entire preprocessing pipeline.

In [ ]:
# Confirm available months in chronological order
available_months = sorted(df['CloseYearMonth'].unique())
print('Available months (chronological):')
for m in available_months:
    print(f'  {m}  ({(df["CloseYearMonth"]==m).sum():,} rows)')

TEST_MONTH = available_months[-1]   # May 2026
print(f'\nTest month (most recent): {TEST_MONTH}')

In [ ]:
def build_train_test_split(df, test_month, window_x, date_col='CloseYearMonth'):
    '''
    Build a train/test split where:
      - test set  = the single most recent month (test_month)
      - train set = the X months immediately preceding test_month

    window_x is intentionally a parameter, not hardcoded, so that different
    training window lengths can be experimented with in Week 4+ to find the
    value of X that yields the best validation performance.
    '''
    available = sorted(df[date_col].unique())
    test_idx = available.index(test_month)

    if test_idx - window_x < 0:
        raise ValueError(
            f'Requested window_x={window_x} months, but only {test_idx} months '
            f'are available before {test_month}.'
        )

    train_months = available[test_idx - window_x: test_idx]

    train_df = df[df[date_col].isin(train_months)].copy()
    test_df  = df[df[date_col] == test_month].copy()

    return train_df, test_df, train_months

In [ ]:
# Example: build the split with the MAXIMUM available window (X=6) as a starting point.
# The optimal X will be determined experimentally in Week 4 by comparing model
# performance across several window lengths (e.g. X=2, 3, 4, 5, 6).
X_WINDOW = 6

train_df, test_df, train_months = build_train_test_split(df, TEST_MONTH, X_WINDOW)

print(f'Training window (X={X_WINDOW} months): {train_months}')
print(f'Training set size: {len(train_df):,} rows')
print(f'Test set size    : {len(test_df):,} rows (month = {TEST_MONTH})')

In [ ]:
# ── Sanity check across several candidate window lengths ─────────
# This does not select the final X (that happens with actual model evaluation
# in Week 4) -- it simply confirms every candidate window is valid and shows
# how training set size changes with X.
print('Training set size for each candidate window length X:')
for x in range(1, 7):
    try:
        _train, _test, _months = build_train_test_split(df, TEST_MONTH, x)
        print(f'  X={x}: {len(_train):>6,} training rows  |  months = {_months}')
    except ValueError as e:
        print(f'  X={x}: {e}')